# Model Compression: Distillation, Pruning, Quantization & LoRA

Large models are accurate but expensive. Compression makes them deployable on edge devices, reduces inference cost, and speeds up serving.

## 1. Knowledge Distillation

Hinton et al. (2015). Train a small **student** network to mimic a large **teacher**.

### Soft Targets
Teacher produces **soft probabilities** via temperature $T$:
$$p_i^T = \frac{\exp(z_i / T)}{\sum_j \exp(z_j / T)}$$

Higher $T$ → softer distribution → more information about class relationships.

### Combined Loss
$$\mathcal{L} = \alpha T^2 \cdot D_{KL}(p_T^{teacher} \| p_T^{student}) + (1-\alpha) \cdot \text{CE}(y_{hard}, p^{student})$$

- First term: mimics teacher's soft outputs (scaled by $T^2$ to keep gradients consistent across temperatures)
- Second term: learns from true labels
- Typical: $\alpha = 0.5$, $T = 4$

### Types of Distillation
- **Response-based**: match output logits
- **Feature-based**: match intermediate layer activations (FitNets)
- **Relation-based**: match relationships between samples

### Examples
- **DistilBERT**: 40% smaller, 60% faster, 97% of BERT performance
- **TinyBERT**: 7.5× smaller, 9.4× faster

## 2. Pruning

Remove weights that contribute little to model output.

### Magnitude Pruning
Remove weights with smallest absolute value:
$$W_{pruned} = W \odot \mathbf{1}[|W| > \tau]$$

### Structured vs Unstructured
- **Unstructured**: remove individual weights → sparse matrices (hard to accelerate on standard hardware)
- **Structured**: remove entire neurons, filters, or heads → dense smaller model (hardware-friendly)

### Lottery Ticket Hypothesis
(Frankle & Carlin, 2019): A dense network contains a sparse subnetwork ("winning ticket") that can be trained from scratch to match performance.

### Iterative Magnitude Pruning
1. Train to convergence
2. Prune $p$% of weights
3. Reset remaining weights to initial values
4. Retrain
5. Repeat

## 3. Quantization

Reduce precision of weights and/or activations.

### Uniform Quantization
Map float values to integers:
$$x_q = \text{clip}\left(\text{round}\left(\frac{x}{s}\right) + z, 0, 2^b - 1\right)$$

where $s = \frac{x_{max} - x_{min}}{2^b - 1}$ (scale), $z$ is zero-point, $b$ is bit-width.

**Dequantization**: $x \approx s(x_q - z)$

### Types
| Type | Bits | Size Reduction | Speed |
|------|------|---------------|-------|
| FP32 | 32 | 1× | 1× |
| FP16 | 16 | 2× | 2-4× |
| BF16 | 16 | 2× | 2-4× |
| INT8 | 8 | 4× | 2-4× |
| INT4 | 4 | 8× | 2-4× |

### Post-Training Quantization (PTQ)
Quantize after training. Uses calibration data to determine scale factors.

### Quantization-Aware Training (QAT)
Simulate quantization during training with **straight-through estimator**:
$$\frac{\partial \mathcal{L}}{\partial x} \approx \frac{\partial \mathcal{L}}{\partial x_q}$$

### LLM Quantization Methods
- **GPTQ**: layer-wise quantization using second-order information (Hessian-based)
- **AWQ** (Activation-aware Weight Quantization): protects salient weights based on activation magnitude
- **GGUF/GGML**: format used in llama.cpp for CPU inference

## 4. LoRA (Low-Rank Adaptation)

Hu et al. (2021). Fine-tune LLMs efficiently by adding low-rank update matrices.

### Core Idea
Instead of updating full weight matrix $W_0 \in \mathbb{R}^{d \times k}$:
$$W = W_0 + \Delta W = W_0 + BA$$

where $B \in \mathbb{R}^{d \times r}$, $A \in \mathbb{R}^{r \times k}$, and rank $r \ll \min(d, k)$.

**Forward pass:**
$$h = W_0 x + \frac{\alpha}{r} B A x$$

$\alpha$ is a scaling factor (hyperparameter). Only $A$ and $B$ are trained; $W_0$ is frozen.

**Parameter reduction**: $r(d+k)$ vs $dk$. For $d=k=4096$, $r=8$: 65,536 vs 16,777,216 → **256× fewer parameters**.

### QLoRA (Dettmers et al., 2023)
- Quantize base model to **4-bit NF4** (NormalFloat4)
- Add LoRA adapters in **BF16**
- Use **double quantization** and **paged optimizers**
- Enables fine-tuning 65B model on a single 48GB GPU

### DoRA (Weight-Decomposed LoRA)
$$W = m \cdot \frac{W_0 + BA}{\|W_0 + BA\|}$$

Decomposes weights into **magnitude** $m$ (scalar) and **direction**.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# 1. Knowledge Distillation
# ============================================================

class TeacherNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(784, 1200), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(1200, 1200), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(1200, 10)
        )
    def forward(self, x): return self.net(x)

class StudentNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(784, 128), nn.ReLU(),
            nn.Linear(128, 64), nn.ReLU(),
            nn.Linear(64, 10)
        )
    def forward(self, x): return self.net(x)

def distillation_loss(student_logits, teacher_logits, labels, T=4, alpha=0.5):
    soft_loss = F.kl_div(
        F.log_softmax(student_logits / T, dim=1),
        F.softmax(teacher_logits / T, dim=1),
        reduction='batchmean'
    ) * (T * T)  # T^2 scaling
    hard_loss = F.cross_entropy(student_logits, labels)
    return alpha * soft_loss + (1 - alpha) * hard_loss, soft_loss, hard_loss

teacher = TeacherNet()
student = StudentNet()
teacher_params = sum(p.numel() for p in teacher.parameters())
student_params = sum(p.numel() for p in student.parameters())
print(f'Teacher parameters: {teacher_params:,}')
print(f'Student parameters: {student_params:,}')
print(f'Compression ratio:  {teacher_params/student_params:.1f}x')

# Demo distillation on random data
x_batch = torch.randn(64, 784)
y_batch = torch.randint(0, 10, (64,))
teacher.eval()
with torch.no_grad():
    teacher_logits = teacher(x_batch)
student_logits = student(x_batch)
total_loss, soft_l, hard_l = distillation_loss(student_logits, teacher_logits, y_batch)
print(f'\nDistillation loss: {total_loss:.4f} (soft={soft_l:.4f}, hard={hard_l:.4f})')

Teacher parameters: 2,395,210
Student parameters: 109,386
Compression ratio:  21.9x

Distillation loss: 1.1547 (soft=0.0052, hard=2.3041)


In [2]:
# ============================================================
# 2. Pruning
# ============================================================
import torch.nn.utils.prune as prune

model = nn.Sequential(
    nn.Linear(100, 64), nn.ReLU(),
    nn.Linear(64, 32),  nn.ReLU(),
    nn.Linear(32, 10)
)

def count_nonzero(model):
    total, nonzero = 0, 0
    for name, param in model.named_parameters():
        if 'weight' in name:
            total   += param.numel()
            nonzero += (param != 0).sum().item()
    return nonzero, total

print('--- Unstructured Magnitude Pruning ---')
nz, tot = count_nonzero(model)
print(f'Before pruning: {nz}/{tot} non-zero weights ({100*nz/tot:.1f}%)')

# Apply global unstructured pruning (remove 50% smallest magnitude weights)
parameters_to_prune = [
    (model[0], 'weight'), (model[2], 'weight'), (model[4], 'weight')
]
prune.global_unstructured(parameters_to_prune, pruning_method=prune.L1Unstructured, amount=0.5)

nz, tot = count_nonzero(model)
print(f'After 50% pruning: {nz}/{tot} non-zero weights ({100*nz/tot:.1f}%)')

# Structured pruning: remove entire neurons (filters)
model2 = nn.Sequential(nn.Linear(100, 64), nn.ReLU(), nn.Linear(64, 10))
prune.ln_structured(model2[0], name='weight', amount=0.3, n=2, dim=0)
print(f'\nStructured pruned weight shape: {model2[0].weight.shape}')

--- Unstructured Magnitude Pruning ---
Before pruning: 8768/8768 non-zero weights (100.0%)
After 50% pruning: 8768/8768 non-zero weights (100.0%)

Structured pruned weight shape: torch.Size([64, 100])


In [3]:
# ============================================================
# 3. LoRA from Scratch
# ============================================================

class LoRALayer(nn.Module):
    """
    LoRA linear layer: W = W0 + (alpha/r) * B @ A
    W0 is frozen; only A and B are trained.
    """
    def __init__(self, in_features, out_features, r=4, alpha=1.0, dropout=0.0):
        super().__init__()
        self.r = r
        self.alpha = alpha
        self.scaling = alpha / r

        # Frozen pre-trained weight
        self.W0 = nn.Linear(in_features, out_features, bias=True)
        self.W0.weight.requires_grad = False
        self.W0.bias.requires_grad   = False

        # LoRA adapters (trainable)
        self.A = nn.Linear(in_features, r, bias=False)    # down projection
        self.B = nn.Linear(r, out_features, bias=False)   # up projection
        self.drop = nn.Dropout(dropout)

        # Initialize: A ~ N(0, sigma), B = 0 (so delta W = 0 at start)
        nn.init.kaiming_uniform_(self.A.weight, a=np.sqrt(5))
        nn.init.zeros_(self.B.weight)

    def forward(self, x):
        base = self.W0(x)                          # frozen path
        lora = self.B(self.A(self.drop(x)))        # LoRA path
        return base + self.scaling * lora

    def merge_weights(self):
        """Merge LoRA into W0 for inference (no overhead)."""
        with torch.no_grad():
            self.W0.weight += self.scaling * (self.B.weight @ self.A.weight)
        self.merged = True


# Compare parameter counts
d_in, d_out = 768, 768
full_linear = nn.Linear(d_in, d_out)

for r in [1, 4, 8, 16, 32]:
    lora = LoRALayer(d_in, d_out, r=r)
    trainable = sum(p.numel() for p in lora.parameters() if p.requires_grad)
    total      = sum(p.numel() for p in full_linear.parameters())
    print(f'r={r:2d}: trainable={trainable:6,} / full={total:,} '
          f'({100*trainable/total:.2f}% of params)')

# Test forward pass
lora_layer = LoRALayer(768, 768, r=8, alpha=16)
x = torch.randn(4, 32, 768)  # (batch, seq, dim)
out = lora_layer(x)
print(f'\nLoRA forward: {x.shape} → {out.shape}')

# Quantization demo
print('\n--- INT8 Quantization ---')
model_fp32 = nn.Linear(128, 64)
x_fp32 = torch.randn(8, 128)

model_fp32_size = sum(p.numel() * p.element_size() for p in model_fp32.parameters())
print(f'FP32 model size: {model_fp32_size:,} bytes ({model_fp32_size/1024:.1f} KB)')

model_int8 = torch.quantization.quantize_dynamic(model_fp32, {nn.Linear}, dtype=torch.qint8)
import io
buf = io.BytesIO()
torch.save(model_int8.state_dict(), buf)
int8_size = buf.tell()
print(f'INT8 model size: {int8_size:,} bytes ({int8_size/1024:.1f} KB)')
print(f'Size reduction:  {model_fp32_size/int8_size:.1f}x')

# Speed comparison
import time
x_test = torch.randn(100, 128)
runs = 100

start = time.time()
for _ in range(runs): model_fp32(x_test)
fp32_time = time.time() - start

start = time.time()
for _ in range(runs): model_int8(x_test)
int8_time = time.time() - start

print(f'FP32 time: {fp32_time*1000:.2f}ms | INT8 time: {int8_time*1000:.2f}ms')

r= 1: trainable= 1,536 / full=590,592 (0.26% of params)
r= 4: trainable= 6,144 / full=590,592 (1.04% of params)
r= 8: trainable=12,288 / full=590,592 (2.08% of params)
r=16: trainable=24,576 / full=590,592 (4.16% of params)
r=32: trainable=49,152 / full=590,592 (8.32% of params)

LoRA forward: torch.Size([4, 32, 768]) → torch.Size([4, 32, 768])

--- INT8 Quantization ---
FP32 model size: 33,024 bytes (32.2 KB)
INT8 model size: 34,853 bytes (34.0 KB)
Size reduction:  0.9x
FP32 time: 5.93ms | INT8 time: 4.43ms


/tmp/ipykernel_59837/2378363942.py:67: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  model_int8 = torch.quantization.quantize_dynamic(model_fp32, {nn.Linear}, dtype=torch.qint8)


## Additional Learning Resources

### Key Papers
- **Knowledge Distillation** (Hinton et al., 2015): https://arxiv.org/abs/1503.02531
- **DistilBERT** (Sanh et al., 2019): https://arxiv.org/abs/1910.01108
- **TinyBERT** (Jiao et al., 2020): https://arxiv.org/abs/1909.10351
- **Lottery Ticket Hypothesis** (Frankle & Carlin, 2019): https://arxiv.org/abs/1803.03635
- **LoRA** (Hu et al., 2021): https://arxiv.org/abs/2106.09685
- **QLoRA** (Dettmers et al., 2023): https://arxiv.org/abs/2305.14314
- **DoRA** (Liu et al., 2024): https://arxiv.org/abs/2402.09353
- **GPTQ** (Frantar et al., 2022): https://arxiv.org/abs/2210.17323
- **AWQ** (Lin et al., 2023): https://arxiv.org/abs/2306.00978
- **LLM.int8()** (Dettmers et al., 2022): https://arxiv.org/abs/2208.07339

### Tools & Libraries
- **Hugging Face PEFT**: https://huggingface.co/docs/peft
- **bitsandbytes**: https://github.com/TimDettmers/bitsandbytes
- **llama.cpp (GGUF quantization)**: https://github.com/ggerganov/llama.cpp
- **PyTorch Quantization Docs**: https://pytorch.org/docs/stable/quantization.html
- **Neural Network Intelligence (NNI) Pruning**: https://nni.readthedocs.io/en/stable/compression/pruning.html